In [ ]:
# Make this notebook runnable from any working directory: locate the repository
# root by its marker file, then work from this notebook's own folder, which is
# what the relative paths below assume.
import os
from pathlib import Path
_root = Path.cwd().resolve()
while not (_root / "requirements.txt").is_file() and _root != _root.parent:
    _root = _root.parent
_here = _root / "refinement-per-claim" / "human-validation-initial"
if not _here.is_dir():
    raise RuntimeError(
        "Could not locate " + str(_here) + ". Run this notebook from inside the "
        "cloned repository."
    )
os.chdir(_here)

# Evaluator 1

In [1]:
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score
import statsmodels.api as sm
from scipy.stats import kendalltau

# Load data
df_1 = pd.read_excel("mini_exp_evaluator_1.xlsx")

# Compute score differences and LLM-predicted preferences
df_1["score_diff"] = df_1["left_score"] - df_1["right_score"]
df_1["llm_pref"] = (df_1["score_diff"] > 0).astype(int)

# Label high/low score difference within each claim-KPI group
def label_difference_level(group):
    group = group.copy()
    group["abs_diff"] = group["score_diff"].abs()
    sorted_group = group.sort_values("abs_diff", ascending=False)
    sorted_group["diff_level"] = ["high"] * 3 + ["low"] * 3
    return sorted_group.drop(columns="abs_diff")

df_1_labeled = df_1.groupby(["claim", "kpi_id"], group_keys=False).apply(label_difference_level)

# Initialize metrics container
metrics = []

# Loop through KPI and difference levels
for kpi in sorted(df_1_labeled["kpi_id"].unique()):
    for level in ["high", "low"]:
        subset = df_1_labeled[(df_1_labeled["kpi_id"] == kpi) & (df_1_labeled["diff_level"] == level)]

        # Accuracy
        accuracy = accuracy_score(subset["left_chosen"], subset["llm_pref"])

        # ROC AUC
        try:
            auc = roc_auc_score(subset["left_chosen"], subset["score_diff"])
        except ValueError:
            auc = None  # AUC undefined if all labels are the same

        # Logistic Regression: human choice ~ score_diff
        X = sm.add_constant(subset["score_diff"])
        y = subset["left_chosen"]
        logit_model = sm.Logit(y, X).fit(disp=0)
        coef = logit_model.params["score_diff"]
        pval = logit_model.pvalues["score_diff"]

        # Kendall’s Tau-b
        tau, tau_p = kendalltau(subset["llm_pref"], subset["left_chosen"])

        metrics.append({
            "kpi_id": kpi,
            "difference_level": level,
            "accuracy": accuracy,
            "roc_auc": auc,
            "logit_coef": coef,
            "logit_p_value": pval,
            "kendall_tau_b": tau,
            "kendall_p_value": tau_p,
            "n_pairs": len(subset)
        })

# Create and print the results table
metrics_df_1 = pd.DataFrame(metrics)
metrics_df_1

,kpi_id,difference_level,accuracy,roc_auc,logit_coef,logit_p_value,kendall_tau_b,kendall_p_value,n_pairs
0,1,high,0.733333,0.741863,0.082048,0.002025,0.461279,0.000395,60
1,1,low,0.416667,0.420249,-0.292955,0.218034,-0.182537,0.160888,60
2,2,high,0.316667,0.305275,-0.036388,0.008128,-0.358313,0.005919,60
3,2,low,0.433333,0.483259,-0.026413,0.905212,-0.129464,0.320012,60
4,3,high,0.300000,0.238857,-0.056795,0.001642,-0.382857,0.003274,60
5,3,low,0.516667,0.510857,-0.008883,0.968471,0.000000,1.000000,60


# Evaluator 2

In [ ]:
# Load data
df_2 = pd.read_excel("mini_exp_evaluator_2.xlsx")

# Compute score differences and LLM-predicted preferences
df_2["score_diff"] = df_2["left_score"] - df_2["right_score"]
df_2["llm_pref"] = (df_2["score_diff"] > 0).astype(int)

# Label high/low score difference within each claim-KPI group
def label_difference_level(group):
    group = group.copy()
    group["abs_diff"] = group["score_diff"].abs()
    sorted_group = group.sort_values("abs_diff", ascending=False)
    sorted_group["diff_level"] = ["high"] * 3 + ["low"] * 3
    return sorted_group.drop(columns="abs_diff")

df_2_labeled = df_2.groupby(["claim", "kpi_id"], group_keys=False).apply(label_difference_level)

# Initialize metrics container
metrics = []

# Loop through KPI and difference levels
for kpi in sorted(df_2_labeled["kpi_id"].unique()):
    for level in ["high", "low"]:
        subset = df_2_labeled[(df_2_labeled["kpi_id"] == kpi) & (df_2_labeled["diff_level"] == level)]

        # Accuracy
        accuracy = accuracy_score(subset["left_chosen"], subset["llm_pref"])

        # ROC AUC
        try:
            auc = roc_auc_score(subset["left_chosen"], subset["score_diff"])
        except ValueError:
            auc = None  # AUC undefined if all labels are the same

        # Logistic Regression: human choice ~ score_diff
        X = sm.add_constant(subset["score_diff"])
        y = subset["left_chosen"]
        logit_model = sm.Logit(y, X).fit(disp=0)
        coef = logit_model.params["score_diff"]
        pval = logit_model.pvalues["score_diff"]

        # Kendall’s Tau-b
        tau, tau_p = kendalltau(subset["llm_pref"], subset["left_chosen"])

        metrics.append({
            "kpi_id": kpi,
            "difference_level": level,
            "accuracy": accuracy,
            "roc_auc": auc,
            "logit_coef": coef,
            "logit_p_value": pval,
            "kendall_tau_b": tau,
            "kendall_p_value": tau_p,
            "n_pairs": len(subset)
        })

# Create and print the results table
metrics_df_2 = pd.DataFrame(metrics)
metrics_df_2

,kpi_id,difference_level,accuracy,roc_auc,logit_coef,logit_p_value,kendall_tau_b,kendall_p_value,n_pairs
0,1,high,0.716667,0.734954,0.079998,0.004153,0.465012,0.000354,60
1,1,low,0.433333,0.476571,-0.099602,0.666593,-0.118918,0.361020,60
2,2,high,0.566667,0.621111,0.019440,0.131424,0.134535,0.301426,60
3,2,low,0.466667,0.503959,0.039245,0.860549,-0.058428,0.653579,60
4,3,high,0.683333,0.674774,0.039449,0.018192,0.352478,0.006781,60
5,3,low,0.583333,0.627920,0.329317,0.147074,0.163390,0.209470,60


# Together

In [ ]:
df_all = pd.concat([df_1, df_2], ignore_index=True)
df_all.head()

In [4]:
# Assume df_all is already a combination of both evaluators' results
df_all["score_diff"] = df_all["left_score"] - df_all["right_score"]
df_all["llm_pref"] = (df_all["score_diff"] > 0).astype(int)

# Label difference level by (claim, kpi_id) only — not per evaluator
def label_difference_level(group):
    group = group.copy()
    group["abs_diff"] = group["score_diff"].abs()

    def assign_diff_level(subgroup):
        sorted_group = subgroup.sort_values("abs_diff", ascending=False)
        sorted_group["diff_level"] = ["high"] * 3 + ["low"] * 3
        return sorted_group

    # Now apply that logic within each evaluator
    labeled = group.groupby("evaluator_id", group_keys=False).apply(assign_diff_level)
    return labeled.drop(columns="abs_diff")

df_labeled = df_all.groupby(["claim", "kpi_id"], group_keys=False).apply(label_difference_level)

# Now compute metrics
metrics = []

for kpi in sorted(df_labeled["kpi_id"].unique()):
    for level in ["high", "low"]:
        subset = df_labeled[(df_labeled["kpi_id"] == kpi) & (df_labeled["diff_level"] == level)]

        # Accuracy
        accuracy = accuracy_score(subset["left_chosen"], subset["llm_pref"])

        # ROC AUC
        try:
            auc = roc_auc_score(subset["left_chosen"], subset["score_diff"])
        except ValueError:
            auc = None

        # Logistic regression
        X = sm.add_constant(subset["score_diff"])
        y = subset["left_chosen"]
        logit_model = sm.Logit(y, X).fit(disp=0)
        coef = logit_model.params["score_diff"]
        pval = logit_model.pvalues["score_diff"]

        # Kendall's Tau-b
        tau, tau_p = kendalltau(subset["llm_pref"], subset["left_chosen"])

        metrics.append({
            "kpi_id": kpi,
            "difference_level": level,
            "accuracy": accuracy,
            "roc_auc": auc,
            "logit_coef": coef,
            "logit_p_value": pval,
            "kendall_tau_b": tau,
            "kendall_p_value": tau_p,
            "n_pairs": len(subset)
        })

metrics_df = pd.DataFrame(metrics)
metrics_df

,kpi_id,difference_level,accuracy,roc_auc,logit_coef,logit_p_value,kendall_tau_b,kendall_p_value,n_pairs
0,1,high,0.725000,0.733083,0.078712,0.000029,0.457865,5.892231e-07,120
1,1,low,0.425000,0.449430,-0.190458,0.242824,-0.149101,1.038426e-01,120
2,2,high,0.441667,0.464077,-0.007527,0.397721,-0.111130,2.254032e-01,120
3,2,low,0.450000,0.493547,0.006181,0.968643,-0.094013,3.051003e-01,120
4,3,high,0.491667,0.458877,-0.006758,0.536894,-0.014088,8.778617e-01,120
5,3,low,0.550000,0.569865,0.159485,0.312961,0.082061,3.706910e-01,120
